# Frame-Level Deepfake Detection using CNN Models

In [ ]:
!pip install timm kaggle --quiet
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [01:33<00:00, 43.1MB/s]



In [ ]:
import torch
import timm
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
from torch.cuda.amp import GradScaler
import random

In [ ]:
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0   # removes classifier safely
        )

        in_features = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 1)
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.classifier(x)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1,0.1,0.1,0.05),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

dataset = datasets.ImageFolder(
    "real_vs_fake/real-vs-fake/train",
    transform=transform
)

subset_indices = random.sample(range(len(dataset)), 20000)
dataset = Subset(dataset, subset_indices)

train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_data, val_data = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepfakeDetector().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
scaler = GradScaler()

def evaluate(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.float().to(device).view(-1,1)

            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

    model.train()
    return total_loss / len(loader)


best_val_loss = float("inf")
patience = 2
patience_counter = 0

for epoch in range(6):


    if epoch < 2:
        for p in model.backbone.parameters():
            p.requires_grad = False
    else:
        for p in model.backbone.parameters():
            p.requires_grad = True

    running_loss = 0

    for i,(imgs,labels) in enumerate(train_loader):

        imgs = imgs.to(device)
        labels = labels.float().to(device).view(-1,1)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda"):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        if i % 150 == 0:
            probs = torch.sigmoid(outputs[:32])
            real_ratio = (probs > 0.5).float().mean().item()

            print(
                f"Epoch {epoch} Batch {i} | "
                f"Loss {loss:.4f} | "
                f"Real {real_ratio:.2f} | "
                f"Fake {1-real_ratio:.2f}"
            )

    train_loss = running_loss / len(train_loader)
    val_loss = evaluate(model, val_loader)

    print(f"Epoch Train Loss: {train_loss}")
    print(f"Validation Loss: {val_loss}")


    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(),"deepfake_b0.pth")
        print("Saved Best Model")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered")
        break


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/tmp/ipykernel_1227/1199738359.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 0 Batch 0 | Loss 0.6844 | Real 0.25 | Fake 0.75
Epoch 0 Batch 150 | Loss 0.4659 | Real 0.56 | Fake 0.44
Epoch Train Loss: 0.4836318245593538
Validation Loss: 0.4181558024138212
Saved Best Model
Epoch 1 Batch 0 | Loss 0.3725 | Real 0.44 | Fake 0.56
Epoch 1 Batch 150 | Loss 0.4429 | Real 0.53 | Fake 0.47
Epoch Train Loss: 0.41034022385769703
Validation Loss: 0.403285619802773
Saved Best Model
Epoch 2 Batch 0 | Loss 0.4385 | Real 0.50 | Fake 0.50
Epoch 2 Batch 150 | Loss 0.2338 | Real 0.34 | Fake 0.66
Epoch Train Loss: 0.1501453805628979
Validation Loss: 0.07461444105138071
Saved Best Model
Epoch 3 Batch 0 | Loss 0.0123 | Real 0.50 | Fake 0.50
Epoch 3 Batch 150 | Loss 0.0110 | Real 0.41 | Fake 0.59
Epoch Train Loss: 0.03573498928321671
Validation Loss: 0.05110408176551573
Saved Best Model
Epoch 4 Batch 0 | Loss 0.0071 | Real 0.66 | Fake 0.34
Epoch 4 Batch 150 | Loss 0.0058 | Real 0.47 | Fake 0.53
Epoch Train Loss: 0.020382607586480386
Validation Loss: 0.035252070738351904
Saved Best

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

print("\nRunning Final Evaluation...")

# Load best saved model
model.load_state_dict(torch.load("deepfake_b0.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)

        outputs = model(imgs)
        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).int().cpu().numpy()

        all_preds.extend(preds.flatten())
        all_labels.extend(labels.numpy())

# Metrics
acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)

print("\n===== FINAL METRICS =====")
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)
print("Confusion Matrix:\n", cm)



Running Final Evaluation...

===== FINAL METRICS =====
Accuracy : 0.988
Precision: 0.9886480908152735
Recall   : 0.9866117404737385
F1 Score : 0.9876288659793815
Confusion Matrix:
 [[1018   11]
 [  13  958]]


In [ ]:
%%writefile df.py

import streamlit as st
import torch
import timm
from torchvision import transforms
from PIL import Image

class DeepfakeDetector(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=False,
            num_classes=0
        )

        in_features = self.backbone.num_features

        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(in_features,512),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.4),
            torch.nn.Linear(512,1)
        )

    def forward(self,x):
        x = self.backbone(x)
        return self.classifier(x)


@st.cache_resource
def load_model():
    model = DeepfakeDetector()
    model.load_state_dict(torch.load("deepfake_b0.pth",map_location="cpu"))
    model.eval()
    return model


st.title("Deepfake Detection Portal")

file = st.file_uploader("Upload Image",type=["jpg","png","jpeg"])

if file:
    image = Image.open(file).convert("RGB")
    st.image(image)

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])])

    model = load_model()

    tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        logit = model(tensor)
        prob = torch.sigmoid(logit).item()

    is_real = prob >= 0.5

    label = "REAL" if is_real else "FAKE"
    confidence = prob if is_real else (1 - prob)

    st.write(f"Prediction: **{label}**")
    st.write(f"Confidence: {confidence:.2%}")

Writing df.py


In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

!pkill streamlit
!pkill cloudflared

import subprocess
import time
with open("logs.txt", "w") as f:
    subprocess.Popen(["streamlit", "run", "df.py", "--server.port", "8501"], stdout=f, stderr=f)

time.sleep(10)

print("--- YOUR APP LINK IS BELOW ---")
!cloudflared tunnel --url http://localhost:8501

--2026-04-08 06:42:57--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-04-08 06:42:57--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-08T07%3A30%3A30Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [ ]:
!pip install streamlit==1.32.0